In [ ]:
!pip install scikit-learn==1.4.2 transformers datasets audiomentations==0.36.0 noisereduce librosa optimum[onnxruntime] onnx onnx-tf tensorflow seaborn matplotlib

import os
import numpy as np
import pandas as pd
import torch
import librosa
import noisereduce as nr
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, TimeMask, AddBackgroundNoise
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from transformers import ASTFeatureExtractor, ASTForAudioClassification, TrainingArguments, Trainer
from datasets import Dataset
from optimum.onnxruntime import ORTModelForAudioClassification
import onnx
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Define paths
train_dir = r"data set path here"
test_dir = r"data set path here"
new_files_dir = r"data set path here"
output_dir = r"put save path here"
log_file = os.path.join(output_dir, "training_errors.log")
failed_files_log = os.path.join(output_dir, "failed_files.csv")
plots_dir = os.path.join(output_dir, "plots")
noise_dir = new_files_dir
failed_files = []

# Create output directories
os.makedirs(output_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)


In [ ]:
augment = Compose([
    AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.05, p=0.7),
    AddBackgroundNoise(sounds_path=noise_dir, min_snr_db=5, max_snr_db=20, p=0.5),
    TimeStretch(min_rate=0.8, max_rate=1.2, p=0.7),
    PitchShift(min_semitones=-4, max_semitones=4, p=0.7),
    TimeMask(min_band_part=0.05, max_band_part=0.2, p=0.5)
])
def preprocess_audio(file_path, sr=16000, duration=3, reduce_noise=True):
    try:
        audio, _ = librosa.load(file_path, sr=44100, duration=duration, mono=True)
        audio = librosa.resample(audio, orig_sr=44100, target_sr=sr)
        if reduce_noise:
            audio = nr.reduce_noise(y=audio, sr=sr, stationary=False, prop_decrease=0.1)
        audio = audio / (np.max(np.abs(audio)) + 1e-9)
        target_length = int(sr * duration)
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)))
        elif len(audio) > target_length:
            audio = audio[:target_length]
        if audio.shape != (target_length,):
            raise ValueError(f"Audio shape {audio.shape} does not match expected ({target_length},)")
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        return audio, np.mean(mfcc, axis=1).tolist()
    except Exception as e:
        logger.error(f"Error preprocessing {file_path}: {e}")
        with open(log_file, "a") as f:
            f.write(f"Error preprocessing {file_path}: {e}\n")
        failed_files.append({'File': file_path, 'Error': str(e)})
        return None, None

In [ ]:
def load_data(directory, sr=16000, augment_data=False):
    X, y, mfcc_means = [], [], []
    class_mapping = {'engine1_good': 'good', 'engine2_broken': 'broken', 'engine3_heavyload': 'heavy_load'}
    file_counts = {cls: 0 for cls in class_mapping.values()}
    failed_files.clear()
    target_length = int(sr * 3)
    
    for folder in ['engine1_good', 'engine2_broken', 'engine3_heavyload']:
        class_dir = os.path.join(directory, folder)
        if not os.path.exists(class_dir):
            logger.warning(f"Directory {class_dir} not found")
            with open(log_file, "a") as f:
                f.write(f"Directory not found: {class_dir}\n")
            continue
        class_label = class_mapping[folder]
        files = [f for f in os.listdir(class_dir) if f.endswith('.wav')]
        file_counts[class_label] = len(files)
        aug_count = 8 if augment_data else 0
        for file in files:
            file_path = os.path.join(class_dir, file)
            audio, mfcc_mean = preprocess_audio(file_path, sr, reduce_noise=class_label != 'heavy_load')
            if audio is None or audio.shape != (target_length,):
                continue
            logger.info(f"Processed {file_path}, shape: {audio.shape}")
            X.append(audio)
            y.append(class_label)
            mfcc_means.append(mfcc_mean)
            if augment_data:
                for i in range(aug_count):
                    try:
                        augmented_audio = audio.copy()
                        for transform in augment.transforms:
                            augmented_audio = transform(samples=augmented_audio, sample_rate=sr)
                        if augmented_audio is None or len(augmented_audio) == 0:
                            logger.warning(f"Augmentation failed for {file_path}, iteration {i}")
                            continue
                        if len(augmented_audio) != target_length:
                            if len(augmented_audio) < target_length:
                                augmented_audio = np.pad(augmented_audio, (0, target_length - len(augmented_audio)))
                            else:
                                augmented_audio = augmented_audio[:target_length]
                        if augmented_audio.shape == (target_length,):
                            X.append(augmented_audio)
                            y.append(class_label)
                            mfcc_aug = librosa.feature.mfcc(y=augmented_audio, sr=sr, n_mfcc=13)
                            mfcc_means.append(np.mean(mfcc_aug, axis=1).tolist())
                            logger.info(f"Augmented {file_path}, iteration {i}, shape: {augmented_audio.shape}")
                        else:
                            logger.warning(f"Invalid augmented audio shape {augmented_audio.shape} for {file_path}, iteration {i}")
                    except Exception as e:
                        logger.error(f"Error augmenting {file_path}, iteration {i}: {e}")
                        with open(log_file, "a") as f:
                            f.write(f"Error augmenting {file_path}, iteration {i}: {e}\n")
                        failed_files.append({'File': file_path, 'Error': f"Augmentation error: {str(e)}"})
    
    if failed_files:
        pd.DataFrame(failed_files).to_csv(failed_files_log, index=False)
        logger.info(f"Failed files logged to {failed_files_log}")
    
    logger.info(f"Dataset sizes: {file_counts}")
    with open(log_file, "a") as f:
        f.write(f"Dataset sizes: {file_counts}\n")
    
    try:
        X_array = np.array(X, dtype=np.float32)
        y_array = np.array(y)
        logger.info(f"X shape: {X_array.shape}, y shape: {y_array.shape}")
        return X_array, y_array, mfcc_means
    except Exception as e:
        logger.error(f"Error converting to NumPy array: {e}")
        with open(log_file, "a") as f:
            f.write(f"Error converting to NumPy array: {e}\n")
        for i, x in enumerate(X):
            if not isinstance(x, np.ndarray) or x.shape != (target_length,):
                logger.error(f"Invalid shape at index {i}: {x.shape if isinstance(x, np.ndarray) else type(x)}")
        exit(1)


In [ ]:
def compute_norm_stats(audio_data, sr=16000):
    specs = []
    for audio in audio_data:
        spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128, fmax=8000)
        spec_db = librosa.power_to_db(spec, ref=np.max)
        specs.append(spec_db.flatten())
    specs = np.concatenate(specs)
    return np.mean(specs), np.std(specs)


In [ ]:
class AudioDataset(Dataset):
    def __init__(self, audio, labels, feature_extractor):
        self.audio = audio
        self.labels = labels
        self.feature_extractor = feature_extractor
        self.label2id = {'good': 0, 'broken': 1, 'heavy_load': 2}
        self.id2label = {v: k for k, v in self.label2id.items()}
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        audio = self.audio[idx]
        label = self.labels[idx]
        inputs = self.feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
        return {
            'input_values': inputs['input_values'].squeeze(0),
            'labels': torch.tensor(self.label2id[label], dtype=torch.long)
        }


In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device)
    
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.savefig(save_path)
    plt.close()
    logger.info(f"Confusion matrix saved to {save_path}")

def plot_training_curves(history, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Accuracy
    train_acc = [m['eval_accuracy'] for m in history if 'eval_accuracy' in m]
    val_acc = [m['eval_accuracy'] for m in history if 'eval_accuracy' in m]
    epochs = range(1, len(train_acc) + 1)
    ax1.plot(epochs, train_acc, label='Training Accuracy')
    ax1.plot(epochs, val_acc, label='Validation Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.set_title('Training and Validation Accuracy')
    ax1.legend()
    
    # Loss
    train_loss = [m['loss'] for m in history if 'loss' in m]
    val_loss = [m['eval_loss'] for m in history if 'eval_loss' in m]
    ax2.plot(epochs, train_loss[:len(val_loss)], label='Training Loss')
    ax2.plot(epochs, val_loss, label='Validation Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Training and Validation Loss')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    logger.info(f"Training curves saved to {save_path}")


In [ ]:
_train, y_train, mfcc_train = load_data(train_dir, augment_data=True)
X_test, y_test, mfcc_test = load_data(test_dir, augment_data=False)

# Encode labels
le = LabelEncoder()
le.classes_ = np.array(['good', 'broken', 'heavy_load'])
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Save MFCC stats
mfcc_df = pd.DataFrame({
    'File': [f"train_{i}" for i in range(len(y_train))] + [f"test_{i}" for i in range(len(y_test))],
    'Label': list(y_train) + list(y_test),
    'MFCC_Mean': mfcc_train + mfcc_test
})
mfcc_df.to_csv(os.path.join(output_dir, "mfcc_stats.csv"), index=False)
logger.info(f"MFCC stats saved to {os.path.join(output_dir, 'mfcc_stats.csv')}")

# Compute normalization stats
mean, std = compute_norm_stats(X_train)
logger.info(f"Computed normalization stats: mean={mean}, std={std}")

# Load feature extractor and model
try:
    feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
    feature_extractor.mean = mean
    feature_extractor.std = std
    model = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593",
        num_labels=3,
        id2label={0: 'good', 1: 'broken', 2: 'heavy_load'},
        label2id={'good': 0, 'broken': 1, 'heavy_load': 2},
        ignore_mismatched_sizes=True
    )
    if torch.cuda.is_available():
        model = model.to('cuda')
        logger.info("Model moved to GPU.")
except Exception as e:
    logger.error(f"Error loading model or feature extractor: {e}")
    with open(log_file, "a") as f:
        f.write(f"Model/feature extractor loading error: {e}\n")
    exit(1)


In [ ]:
# Prepare datasets
train_dataset = AudioDataset(X_train, y_train, feature_extractor)
test_dataset = AudioDataset(X_test, y_test, feature_extractor)
train_hf_dataset = Dataset.from_dict({
    'input_values': [train_dataset[i]['input_values'].numpy() for i in range(len(train_dataset))],
    'labels': [train_dataset[i]['labels'].item() for i in range(len(train_dataset))]
})
test_hf_dataset = Dataset.from_dict({
    'input_values': [test_dataset[i]['input_values'].numpy() for i in range(len(test_dataset))],
    'labels': [test_dataset[i]['labels'].item() for i in range(len(test_dataset))]
})

# Compute class weights
class_counts = np.bincount(y_train_encoded)
total_samples = len(y_train_encoded)
class_weights = torch.tensor([total_samples / (len(class_counts) * count) for count in class_counts], dtype=torch.float32)

# Define compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

# Hyperparameter tuning
learning_rates = [1e-5, 2e-5, 5e-5]
batch_sizes = [8, 16]
best_accuracy = 0
best_model_path = None
best_params = None
history = []

for lr in learning_rates:
    for bs in batch_sizes:
        try:
            training_args = TrainingArguments(
                output_dir=os.path.join(output_dir, f"run_lr{lr}_bs{bs}"),
                evaluation_strategy="epoch",
                save_strategy="epoch",
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                num_train_epochs=5,
                warmup_ratio=0.1,
                logging_steps=10,
                load_best_model_at_end=True,
                metric_for_best_model="accuracy",
                fp16=True,
                report_to="none"
            )
            
            trainer = CustomTrainer(
                model=model,
                args=training_args,
                train_dataset=train_hf_dataset,
                eval_dataset=test_hf_dataset,
                compute_metrics=compute_metrics,
                class_weights=class_weights
            )
            
            train_result = trainer.train()
            eval_result = trainer.evaluate()
            history.extend(trainer.state.log_history)
            
            if eval_result['eval_accuracy'] > best_accuracy:
                best_accuracy = eval_result['eval_accuracy']
                best_model_path = os.path.join(output_dir, f"run_lr{lr}_bs{bs}", "checkpoint-best")
                best_params = {'lr': lr, 'bs': bs}
                trainer.save_model(best_model_path)
            
            logger.info(f"Completed run with lr={lr}, bs={bs}, accuracy={eval_result['eval_accuracy']}")
        except Exception as e:
            logger.error(f"Error in training with lr={lr}, bs={bs}: {e}")
            with open(log_file, "a") as f:
                f.write(f"Training error with lr={lr}, bs={bs}: {e}\n")

# Load best model
if best_model_path:
    model = ASTForAudioClassification.from_pretrained(best_model_path)
    if torch.cuda.is_available():
        model = model.to('cuda')
    trainer = CustomTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=best_model_path,
            per_device_eval_batch_size=best_params['bs'],
            fp16=True,
            report_to="none"
        ),
        eval_dataset=test_hf_dataset,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )
    
    # Evaluate and plot
    predictions = trainer.predict(test_hf_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = predictions.label_ids
    
    # Confusion matrix
    plot_confusion_matrix(y_true, y_pred, le.classes_, os.path.join(plots_dir, "confusion_matrix.png"))
    
    # Training curves
    plot_training_curves(history, os.path.join(plots_dir, "training_curves.png"))
    
    # Classification report
    report = classification_report(y_true, y_pred, target_names=le.classes_, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(os.path.join(output_dir, "classification_report.csv"))
    logger.info(f"Classification report saved to {os.path.join(output_dir, 'classification_report.csv')}")
else:
    logger.error("No successful training runs completed.")
    exit(1)


In [ ]:
# Export to ONNX
onnx_path = os.path.join(output_dir, "model.onnx")
try:
    ort_model = ORTModelForAudioClassification.from_pretrained(best_model_path, export=True)
    ort_model.save_pretrained(os.path.join(output_dir, "onnx_model"))
    logger.info(f"Model exported to ONNX: {onnx_path}")
except Exception as e:
    logger.error(f"Error exporting to ONNX: {e}")
    with open(log_file, "a") as f:
        f.write(f"ONNX export error: {e}\n")
    exit(1)

# Convert ONNX to TFLite
tflite_path = os.path.join(output_dir, "model.tflite")
try:
    # Create representative dataset for quantization
    def representative_dataset():
        for i in range(min(100, len(X_test))):
            inputs = feature_extractor(X_test[i], sampling_rate=16000, return_tensors="np")['input_values']
            yield [inputs.astype(np.float32)]
    
    # Load ONNX model
    onnx_model = onnx.load(onnx_path)
    
    # Convert to TensorFlow
    from onnx_tf.backend import prepare
    tf_rep = prepare(onnx_model)
    tf_model_path = os.path.join(output_dir, "tf_model")
    tf_rep.export_graph(tf_model_path)
    
    # Convert to TFLite with quantization
    converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_model = converter.convert()
    
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    logger.info(f"Model converted to TFLite: {tflite_path}")
except Exception as e:
    logger.error(f"Error converting to TFLite: {e}")
    with open(log_file, "a") as f:
        f.write(f"TFLite conversion error: {e}\n")
    exit(1)

# Test TFLite model
def test_tflite_model(tflite_path, X_test, y_test, feature_extractor):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    correct = 0
    total = 0
    y_pred_tflite = []
    
    for i in range(len(X_test)):
        inputs = feature_extractor(X_test[i], sampling_rate=16000, return_tensors="np")['input_values']
        inputs = (inputs * 127.5).astype(np.int8)  # Quantize to int8
        interpreter.set_tensor(input_details[0]['index'], inputs)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        pred = np.argmax(output, axis=-1)
        y_pred_tflite.append(pred.item())
        if pred == y_test[i]:
            correct += 1
        total += 1
    
    accuracy = correct / total
    logger.info(f"TFLite model accuracy: {accuracy}")
    
    # TFLite confusion matrix
    plot_confusion_matrix(y_test, y_pred_tflite, le.classes_, os.path.join(plots_dir, "confusion_matrix_tflite.png"))

test_tflite_model(tflite_path, X_test, y_test_encoded, feature_extractor)

# Inference on new .wav files
new_predictions = []
for file in Path(new_files_dir).glob("*.wav"):
    try:
        audio, _ = librosa.load(file, sr=16000, duration=3, mono=True)
        inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            if torch.cuda.is_available():
                inputs = {k: v.to('cuda') for k, v in inputs.items()}
            logits = model(**inputs).logits
        pred_id = torch.argmax(logits, dim=-1).item()
        pred_label = model.config.id2label[pred_id]
        new_predictions.append({'File': str(file), 'Prediction': pred_label})
        logger.info(f"Predicted {file}: {pred_label}")
    except Exception as e:
        logger.error(f"Error predicting {file}: {e}")
        with open(log_file, "a") as f:
            f.write(f"Prediction error for {file}: {e}\n")

if new_predictions:
    pd.DataFrame(new_predictions).to_csv(os.path.join(output_dir, "new_predictions.csv"), index=False)
    logger.info(f"New predictions saved to {os.path.join(output_dir, 'new_predictions.csv')}")